# LLM-Backed Semantic Communication (T5 + AWGN)

This notebook is separate from your seq2seq baseline and focuses on an LLM-backed pipeline:
1. Load local Europarl English text
2. Build T5-based semantic communication model
3. Inject AWGN in encoder latent space
4. Train on identity reconstruction
5. Evaluate one sentence across multiple SNR values

In [5]:
# Run once if packages are missing (self-healing in current kernel)
import sys
import subprocess

def ensure_package(pkg_name):
    try:
        __import__(pkg_name)
        return
    except ModuleNotFoundError:
        print(f'Installing missing package: {pkg_name}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])

# Core deps for this notebook
ensure_package('transformers')
ensure_package('sentencepiece')
ensure_package('accelerate')
ensure_package('jiwer')
ensure_package('nltk')

import random
import math
import copy
from pathlib import Path
from typing import List

import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.modeling_outputs import BaseModelOutput
from jiwer import wer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


## Environment check (clean reproducibility)
Run the next cell to print exact versions and hardware info used for this notebook.

In [6]:
# Version + hardware snapshot
import sys
import platform
import torch
import transformers

try:
    import sentencepiece as spm
    sentencepiece_version = spm.__version__
except Exception:
    sentencepiece_version = 'not installed'

print('Python         :', sys.version.split()[0])
print('Platform       :', platform.platform())
print('PyTorch        :', torch.__version__)
print('Transformers   :', transformers.__version__)
print('SentencePiece  :', sentencepiece_version)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device    :', torch.cuda.get_device_name(0))

Python         : 3.11.14
Platform       : Windows-10-10.0.26200-SP0
PyTorch        : 2.5.1
Transformers   : 5.3.0
SentencePiece  : 0.2.1
CUDA available : False


In [7]:
# Hugging Face local cache setup (project-local)
import os
from pathlib import Path

HF_CACHE_DIR = Path('hf_cache').resolve()
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Keep cache local to this project
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE_DIR)

# Optional: suppress Windows symlink warning
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

print('HF cache directory:', HF_CACHE_DIR)
print('Tip: set HF_TOKEN env var for higher download limits if needed.')

HF cache directory: C:\PROJECTS\BTP_Thesis work\hf_cache
Tip: set HF_TOKEN env var for higher download limits if needed.


## Quick T5 download smoke test (from your setup notes)
Run the next cell once to verify your environment can load `t5-small` locally.
- It uses `transformers + torch + sentencepiece`.
- First run downloads model/tokenizer into local cache.
- If this works, the rest of this notebook will work.

In [8]:
# T5-small smoke test: local-cache download + generation (self-contained)
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
from pathlib import Path

# Fallback: define local cache dir if previous setup cell was not run
if 'HF_CACHE_DIR' not in globals():
    HF_CACHE_DIR = Path('hf_cache').resolve()
    HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

tokenizer_smoke = T5Tokenizer.from_pretrained('t5-small', cache_dir=str(HF_CACHE_DIR))
model_smoke = T5ForConditionalGeneration.from_pretrained('t5-small', cache_dir=str(HF_CACHE_DIR))

device_smoke = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_smoke.to(device_smoke)
print('Smoke test device:', device_smoke)
print('Smoke test cache :', HF_CACHE_DIR)

input_text = (
    'summarize: The Text-to-Text Transfer Transformer frames NLP tasks as text-to-text. '
    'It can be adapted for summarization, translation, and question answering.'
)

inputs = tokenizer_smoke(
    input_text,
    return_tensors='pt',
    max_length=256,
    truncation=True,
    padding='max_length'
)['input_ids'].to(device_smoke)

summary_ids = model_smoke.generate(
    inputs,
    max_length=40,
    min_length=8,
    num_beams=4,
    early_stopping=True,
    no_repeat_ngram_size=2
)

summary = tokenizer_smoke.decode(summary_ids[0], skip_special_tokens=True)
print('Smoke test output:', summary if summary.strip() else '[empty output]')

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 2518.07it/s]


Smoke test device: cpu
Smoke test cache : C:\PROJECTS\BTP_Thesis work\hf_cache
Smoke test output:  the Text-to-Text Transfer Transformer can be adapted for summarization, translation, and question answering. the Transformer is available in the u.s.


In [9]:
# Local Europarl sentence loader
LOCAL_EUROPARL_DIR = Path('europarl/en/en')
MAX_SAMPLES = 20000

def _read_text_file(path: Path) -> str:
    try:
        return path.read_text(encoding='utf-8')
    except UnicodeDecodeError:
        return path.read_text(encoding='latin-1', errors='ignore')

def _extract_lines_from_text(raw_text: str):
    lines = []
    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith('<') and line.endswith('>'):
            continue
        lines.append(line)
    return lines

def load_sentences_from_local(folder: Path, max_samples: int = 20000):
    txt_files = sorted(folder.glob('*.txt'))
    if not txt_files:
        raise FileNotFoundError(f'No .txt files found in: {folder.resolve()}')

    sentences = []
    for file_path in txt_files:
        raw = _read_text_file(file_path)
        sentences.extend(_extract_lines_from_text(raw))
        if len(sentences) >= max_samples:
            break

    sentences = [s for s in sentences if len(s.split()) >= 4]
    return sentences[:max_samples]

sentences = load_sentences_from_local(LOCAL_EUROPARL_DIR, MAX_SAMPLES)
print('Loaded sentences:', len(sentences))
print('Sample:', sentences[0])

Loaded sentences: 20000
Sample: Resumption of the session


In [10]:
# Train/validation split
SEED = 42
random.seed(SEED)
shuffled = sentences[:]
random.shuffle(shuffled)
split_idx = int(0.9 * len(shuffled))
train_texts = shuffled[:split_idx]
val_texts = shuffled[split_idx:]

print('Train:', len(train_texts), '| Val:', len(val_texts))

Train: 18000 | Val: 2000


## Full fine-tune vs lightweight tuning
- Set `FULL_FINETUNE = True` for full model tuning (higher compute).
- Set `FULL_FINETUNE = False` to train only channel adapters (`tx`, `rx`) for faster local experimentation.

In [11]:
class LLMChannelModel(nn.Module):
    def __init__(self, model_name='t5-small', full_finetune=False):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.d_model = self.model.config.d_model
        self.full_finetune = full_finetune

        self.tx = nn.Linear(self.d_model, self.d_model)
        self.rx = nn.Linear(self.d_model, self.d_model)

        if not self.full_finetune:
            for parameter in self.model.parameters():
                parameter.requires_grad = False

    def awgn(self, x, snr_db: float):
        signal_power = torch.mean(x.pow(2))
        snr_linear = 10 ** (snr_db / 10.0)
        noise_power = signal_power / snr_linear
        noise_std = torch.sqrt(noise_power + 1e-12)
        noise = noise_std * torch.randn_like(x)
        return x + noise

    def _encode_noisy(self, input_ids, attention_mask, snr_db: float):
        enc = self.model.get_encoder()(input_ids=input_ids, attention_mask=attention_mask)
        hidden = self.tx(enc.last_hidden_state)
        hidden = self.awgn(hidden, snr_db=snr_db)
        hidden = self.rx(hidden)
        return BaseModelOutput(last_hidden_state=hidden)

    def loss_on_batch(self, src_texts: List[str], tgt_texts: List[str], snr_db: float, max_len=128):
        tok = self.tokenizer(src_texts, padding=True, truncation=True, max_length=max_len, return_tensors='pt')

        input_ids = tok['input_ids'].to(device)
        attention_mask = tok['attention_mask'].to(device)

        # Fast path for identity reconstruction: avoid second tokenizer call
        if src_texts == tgt_texts:
            labels = input_ids.clone()
        else:
            tgt = self.tokenizer(text_target=tgt_texts, padding=True, truncation=True, max_length=max_len, return_tensors='pt')
            labels = tgt['input_ids'].to(device)

        labels[labels == self.tokenizer.pad_token_id] = -100

        enc_noisy = self._encode_noisy(input_ids, attention_mask, snr_db=snr_db)
        out = self.model(encoder_outputs=enc_noisy, attention_mask=attention_mask, labels=labels)
        return out.loss

    @torch.no_grad()
    def generate_text(self, src_text: str, snr_db: float, max_len=128):
        tok = self.tokenizer([src_text], padding=True, truncation=True, max_length=max_len, return_tensors='pt')
        input_ids = tok['input_ids'].to(device)
        attention_mask = tok['attention_mask'].to(device)
        enc_noisy = self._encode_noisy(input_ids, attention_mask, snr_db=snr_db)
        gen_ids = self.model.generate(encoder_outputs=enc_noisy, attention_mask=attention_mask, max_length=max_len)
        return self.tokenizer.batch_decode(gen_ids, skip_special_tokens=True)[0]

FULL_FINETUNE = False
llm_semcomm = LLMChannelModel(model_name='t5-small', full_finetune=FULL_FINETUNE).to(device)

trainable = sum(p.numel() for p in llm_semcomm.parameters() if p.requires_grad)
total = sum(p.numel() for p in llm_semcomm.parameters())
print(f'LLM ready | full_finetune={FULL_FINETUNE} | trainable={trainable:,} / total={total:,}')

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 3376.29it/s]


LLM ready | full_finetune=False | trainable=525,312 / total=61,031,936


In [12]:
# Training (with fast precheck mode to avoid long stalls)
def make_text_batches(texts, batch_size=8):
    for idx in range(0, len(texts), batch_size):
        yield texts[idx:idx+batch_size]

# ---------- Run presets ----------
# quick: for sanity checks before demos
# mentor: stable quality with manageable runtime
TRAIN_RUN_MODE = 'quick'  # change to 'mentor' when needed

if TRAIN_RUN_MODE == 'quick':
    TRAIN_TEXTS_LLM = train_texts[:800]
    VAL_TEXTS_LLM = val_texts[:120]
    EPOCHS_LLM = 1
    BATCH_SIZE_LLM = 4
    MAX_LEN_TRAIN = 64
else:
    TRAIN_TEXTS_LLM = train_texts[:4000]
    VAL_TEXTS_LLM = val_texts[:500]
    EPOCHS_LLM = 3
    BATCH_SIZE_LLM = 8
    MAX_LEN_TRAIN = 96

TRAIN_SNR_DB_LLM = 10.0
LR_LLM = 2e-4 if FULL_FINETUNE else 1e-3
PRECHECK_BATCHES = 2

print(f'Train mode={TRAIN_RUN_MODE} | train={len(TRAIN_TEXTS_LLM)} | val={len(VAL_TEXTS_LLM)} | epochs={EPOCHS_LLM} | batch={BATCH_SIZE_LLM} | max_len={MAX_LEN_TRAIN}')

optimizer_llm = torch.optim.AdamW((p for p in llm_semcomm.parameters() if p.requires_grad), lr=LR_LLM)
best_val = float('inf')
best_state = None

# ---------- Tiny precheck ----------
llm_semcomm.train()
print('Running training precheck...')
for step_idx, batch_texts in enumerate(make_text_batches(TRAIN_TEXTS_LLM, batch_size=BATCH_SIZE_LLM), start=1):
    optimizer_llm.zero_grad()
    loss = llm_semcomm.loss_on_batch(batch_texts, batch_texts, snr_db=TRAIN_SNR_DB_LLM, max_len=MAX_LEN_TRAIN)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(llm_semcomm.parameters(), 1.0)
    optimizer_llm.step()
    print(f'Precheck batch {step_idx}/{PRECHECK_BATCHES} | loss={float(loss.detach().cpu()):.4f}')
    if step_idx >= PRECHECK_BATCHES:
        break

# ---------- Full training ----------
for epoch in range(1, EPOCHS_LLM + 1):
    llm_semcomm.train()
    tr_loss = 0.0
    tr_steps = 0

    random.shuffle(TRAIN_TEXTS_LLM)
    for batch_texts in make_text_batches(TRAIN_TEXTS_LLM, batch_size=BATCH_SIZE_LLM):
        optimizer_llm.zero_grad()
        loss = llm_semcomm.loss_on_batch(batch_texts, batch_texts, snr_db=TRAIN_SNR_DB_LLM, max_len=MAX_LEN_TRAIN)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(llm_semcomm.parameters(), 1.0)
        optimizer_llm.step()

        tr_loss += float(loss.detach().cpu())
        tr_steps += 1

    llm_semcomm.eval()
    va_loss = 0.0
    va_steps = 0
    with torch.no_grad():
        for batch_texts in make_text_batches(VAL_TEXTS_LLM, batch_size=BATCH_SIZE_LLM):
            loss = llm_semcomm.loss_on_batch(batch_texts, batch_texts, snr_db=TRAIN_SNR_DB_LLM, max_len=MAX_LEN_TRAIN)
            va_loss += float(loss.detach().cpu())
            va_steps += 1

    tr_avg = tr_loss / max(tr_steps, 1)
    va_avg = va_loss / max(va_steps, 1)

    if va_avg < best_val:
        best_val = va_avg
        best_state = copy.deepcopy(llm_semcomm.state_dict())

    print(f'LLM Epoch {epoch:02d}/{EPOCHS_LLM} | train_loss={tr_avg:.4f} | val_loss={va_avg:.4f} | best_val={best_val:.4f}')

if best_state is not None:
    llm_semcomm.load_state_dict(best_state)
    print('Restored best LLM checkpoint.')

Train mode=quick | train=800 | val=120 | epochs=1 | batch=4 | max_len=64
Running training precheck...
Precheck batch 1/2 | loss=5.2295
Precheck batch 2/2 | loss=4.6096
LLM Epoch 01/1 | train_loss=2.5895 | val_loss=0.2835 | best_val=0.2835
Restored best LLM checkpoint.


In [13]:
# Quick communication examples (few real sentences across SNRs)
import pandas as pd

pd.set_option('display.max_colwidth', None)

DEMO_SENT_COUNT = 3
DEMO_SNR_LIST = [0.0, 5.0, 10.0, 15.0]
DEMO_MAX_LEN = 72
DEMO_SEED = 99

rng_demo = random.Random(DEMO_SEED)
pool_demo = val_texts if len(val_texts) > 0 else sentences
demo_texts = rng_demo.sample(pool_demo, min(DEMO_SENT_COUNT, len(pool_demo)))

smoothie = SmoothingFunction().method1
demo_rows = []

for sent_id, ref_text in enumerate(demo_texts, start=1):
    for snr_db in DEMO_SNR_LIST:
        pred_text = llm_semcomm.generate_text(ref_text, snr_db=snr_db, max_len=DEMO_MAX_LEN)

        ref_tokens = ref_text.split()
        hyp_tokens = pred_text.split()
        bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothie) if hyp_tokens else 0.0
        err = wer([ref_text], [pred_text])

        demo_rows.append({
            'sent_id': sent_id,
            'snr_db': snr_db,
            'reference_text': ref_text,
            'predicted_text': pred_text,
            'bleu': round(float(bleu), 3),
            'wer': round(float(err), 3),
        })

df_demo = pd.DataFrame(demo_rows)
display(df_demo)

summary_demo = (
    df_demo.groupby('snr_db', as_index=False)
    .agg(bleu_mean=('bleu', 'mean'), wer_mean=('wer', 'mean'))
)
summary_demo['bleu_mean'] = summary_demo['bleu_mean'].round(3)
summary_demo['wer_mean'] = summary_demo['wer_mean'].round(3)
print('Quick example summary by SNR:')
display(summary_demo)

,sent_id,snr_db,reference_text,predicted_text,bleu,wer
0,1,0.0,There are a few of the EPP-ED amendments which I can accept although it is perhaps a pity that they did not make more input at committee stage. Could I appeal to Members of that group to deal with their points of disagreement by making explanations of vote. I believe this report will get majority support here in plenary. It was adopted by a large majority in committee and some of the supporters of the Conservatives and Christian Democrats may not appreciate their failure to be part of that majority.,There are some of the EPP-ED Amendments which I can accept by a'rapast of their input at committee stage. Could I appeal to members of that Group to deal with their points of disagreement by taking explanations of their report here in plen. I believe this report will get majority in plen.,0.243,0.622
1,1,5.0,There are a few of the EPP-ED amendments which I can accept although it is perhaps a pity that they did not make more input at committee stage. Could I appeal to Members of that group to deal with their points of disagreement by making explanations of vote. I believe this report will get majority support here in plenary. It was adopted by a large majority in committee and some of the supporters of the Conservatives and Christian Democrats may not appreciate their failure to be part of that majority.,There are several of the EPD-ED Amendments which I can accept although they did not make more input at Committee stage. I could appeal to members of that Group to deal with their points of disagreement by making explanations of vote. I believe this report will get majority support here in plen.,0.307,0.522
2,1,10.0,There are a few of the EPP-ED amendments which I can accept although it is perhaps a pity that they did not make more input at committee stage. Could I appeal to Members of that group to deal with their points of disagreement by making explanations of vote. I believe this report will get majority support here in plenary. It was adopted by a large majority in committee and some of the supporters of the Conservatives and Christian Democrats may not appreciate their failure to be part of that majority.,There are some of the EMP-ED Amendments which I can accept although it is perhaps a sigh that they did not make more input at committee stage. Could I appeal to members of that Group to deal with their points of disagreement by making explanations of vote. I believe this report will get majority support here in plen,0.439,0.433
3,1,15.0,There are a few of the EPP-ED amendments which I can accept although it is perhaps a pity that they did not make more input at committee stage. Could I appeal to Members of that group to deal with their points of disagreement by making explanations of vote. I believe this report will get majority support here in plenary. It was adopted by a large majority in committee and some of the supporters of the Conservatives and Christian Democrats may not appreciate their failure to be part of that majority.,There are some of the EPP-ED Amendments which I can accept although it is perhaps a sigh that they did not make more input at committee stage. Could I appeal to members of that Group to deal with their points of disagreement by making explanations of vote. I believe this report will get majority support here in plen,0.446,0.422
4,2,0.0,"For us, a transitional period until 2006, as proposed by the Commission, is not acceptable, and we should wait at least until 2010 before deciding on possible reforms.","For us, a transitional period between 2006, as proposed by the Commission, is not acceptable, and we should wait at least",0.620,0.286
5,2,5.0,"For us, a transitional period until 2006, as proposed by the Commission, is not acceptable, and we should wait at least until 2010 before deciding on possible reforms.","For us, a transitional period between 2006 and 2006, as proposed by the Commission, is not acceptable, and we should wait at lea

Quick example summary by SNR:


,snr_db,bleu_mean,wer_mean
0,0.0,0.310,0.550
1,5.0,0.378,0.464
2,10.0,0.477,0.381
3,15.0,0.479,0.377


## Realistic evaluation (many sentences)
Single-sentence BLEU/WER can be overly optimistic.
Use the next cell to evaluate across many validation sentences and report averaged metrics per SNR.

## Mentor Demo Flow (run in this order)
1. Quick communication examples (few real sentences across SNR)
2. NLP critical-unit verification (numbers, negation, intent words)
3. Full aggregate benchmark

Design choice: every long test starts with a tiny 2-sentence precheck so you catch failures early and save time.

*Verifying that decision-critical semantic units (numbers, negations, domain keywords) survive the noisy channel*

What I did to solve this issue-

`Use 5 hand-crafted test cases that deliberately include these units so we can reliably test whether they're preserved`

In [14]:
# Practical NLP-level signal verification (real validation sentences across SNR)
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', None)

NEGATION_WORDS = {'not', 'no', 'never', "don't", "doesn't", "didn't", "can't", "won't", "isn't", "aren't", "without"}
DOMAIN_KEYWORDS = {
    'approve', 'approved', 'reject', 'rejected', 'budget', 'payment', 'invoice', 'deadline',
    'delay', 'meeting', 'contract', 'risk', 'security', 'urgent', 'refund', 'cancel'
}

def normalize_words(text: str):
    return re.findall(r"[a-zA-Z']+", text.lower())

def extract_numbers(text: str):
    return set(re.findall(r"\b\d+(?:\.\d+)?%?\b", text))

def extract_critical_units(text: str):
    words = set(normalize_words(text))
    numbers = extract_numbers(text)
    negations = words.intersection(NEGATION_WORDS)
    keywords = words.intersection(DOMAIN_KEYWORDS)
    return numbers.union(negations).union(keywords)

def normalize_numeric_token(token: str):
    return token.replace(' ', '').replace(',', '')

def critical_recall(reference: str, hypothesis: str):
    ref_units = extract_critical_units(reference)
    hyp_units = extract_critical_units(hypothesis)
    ref_norm = {normalize_numeric_token(x) for x in ref_units}
    hyp_norm = {normalize_numeric_token(x) for x in hyp_units}
    if len(ref_norm) == 0:
        return 1.0, sorted(ref_units), sorted([])
    overlap = ref_norm.intersection(hyp_norm)
    matched_original = sorted([u for u in ref_units if normalize_numeric_token(u) in overlap])
    return len(overlap) / len(ref_norm), sorted(ref_units), matched_original

# Sample 5 random validation sentences (keep output readable: filter for medium-length)
VERIFY_SEED = 555
MIN_WORDS = 8
MAX_WORDS = 40

pool_verify = val_texts if len(val_texts) > 0 else sentences
short_sents = [s for s in pool_verify if MIN_WORDS <= len(s.split()) <= MAX_WORDS]

rng_verify = random.Random(VERIFY_SEED)
TEST_CASES = rng_verify.sample(short_sents, min(5, len(short_sents)))

print(f'Verification sentences (random seed={VERIFY_SEED}, length: {MIN_WORDS}-{MAX_WORDS} words):')
for i, sent in enumerate(TEST_CASES, 1):
    print(f'{i}. [{len(sent.split())} words] {sent}')
print()

snr_list_verify = [0.0, 5.0, 10.0, 15.0]
smoothie = SmoothingFunction().method1
rows_verify = []

# Frame: sentence_id, snr_db, reference, predicted, metrics
for sent_id, ref_text in enumerate(TEST_CASES, start=1):
    for snr_db in snr_list_verify:
        pred_text = llm_semcomm.generate_text(ref_text, snr_db=snr_db, max_len=72)

        ref_tokens = ref_text.split()
        hyp_tokens = pred_text.split()
        bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothie) if hyp_tokens else 0.0
        err = wer([ref_text], [pred_text])

        crit_recall, crit_ref, crit_hit = critical_recall(ref_text, pred_text)
        
        rows_verify.append({
            'sent_id': sent_id,
            'snr_db': snr_db,
            'reference': ref_text,
            'predicted': pred_text,
            'bleu': round(float(bleu), 3),
            'wer': round(float(err), 3),
            'critical_recall': round(float(crit_recall), 3),
            'signal_ok': crit_recall >= 0.8,
        })

df_verify = pd.DataFrame(rows_verify)

# Display table: compact format - show key columns only
compact_cols = ['sent_id', 'snr_db', 'reference', 'predicted', 'bleu', 'wer', 'critical_recall', 'signal_ok']
df_compact = df_verify[compact_cols].copy()
df_compact['reference'] = df_compact['reference'].str[:60] + '...'
df_compact['predicted'] = df_compact['predicted'].str[:60] + '...'

print('Verification Results (by sentence):')
display(df_compact.sort_values(['sent_id', 'snr_db']))

print('\nPer-SNR Summary:')
summary_verify = (
    df_verify.groupby('snr_db', as_index=False)
    .agg({
        'critical_recall': 'mean',
        'bleu': 'mean',
        'wer': 'mean',
        'signal_ok': 'mean',
    })
    .round(3)
    .rename(columns={'critical_recall': 'crit_recall_mean', 'bleu': 'bleu_mean', 'wer': 'wer_mean', 'signal_ok': 'ok_rate'})
)
display(summary_verify)

Verification sentences (random seed=555, length: 8-40 words):
1. [26 words] However, there is no guarantee that the chocolate industry will use shea nut oil which, in any case, can only replace 50% of the cocoa butter.
2. [31 words] What else will the Union undertake in this respect if the demand from the opposition - bringing forward elections - is met? What else can we do to support this opposition?
3. [16 words] - It is now sixteen years since the debate on chocolate started in the European Parliament.
4. [31 words] We would like to make three points. Firstly, the active recognition of the role, protagonism and participation of the regions in the process of European construction which began 50 years ago.
5. [15 words] We also cannot accept the individualisation of human rights to be found in this report.

Verification Results (by sentence):


,sent_id,snr_db,reference,predicted,bleu,wer,critical_recall,signal_ok
0,1,0.0,"However, there is no guarantee that the chocolate industry w...","However, there is no guarantee that the chocolate industry w...",0.584,0.269,0.5,False
1,1,5.0,"However, there is no guarantee that the chocolate industry w...","However, there is no guarantee that the chocolate industry w...",1.000,0.000,1.0,True
2,1,10.0,"However, there is no guarantee that the chocolate industry w...","However, there is no guarantee that the chocolate industry w...",1.000,0.000,1.0,True
3,1,15.0,"However, there is no guarantee that the chocolate industry w...","However, there is no guarantee that the chocolate industry w...",1.000,0.000,1.0,True
4,2,0.0,What else will the Union undertake in this respect if the de...,What else will the Union undertake in this regard if the dem...,0.731,0.097,1.0,True
5,2,5.0,What else will the Union undertake in this respect if the de...,What else will the Union undertake in this regard if the dem...,0.913,0.032,1.0,True
6,2,10.0,What else will the Union undertake in this respect if the de...,What else will the Union undertake in this regard if the dem...,0.913,0.032,1.0,True
7,2,15.0,What else will the Union undertake in this respect if the de...,What else will the Union undertake in this regard if the dem...,0.913,0.032,1.0,True
8,3,0.0,- It is now sixteen years since the debate on chocolate star...,- It is now 16 years since the debate on chocolate began in ...,0.613,0.125,1.0,True
9,3,5.0,- It is now sixteen years since the debate on chocolate star...,- It is now sixteen years since the debate on chocolate star...,1.000,0.000,1.0,True



Per-SNR Summary:


,snr_db,crit_recall_mean,bleu_mean,wer_mean,ok_rate
0,0.0,0.7,0.574,0.364,0.6
1,5.0,1.0,0.901,0.039,1.0
2,10.0,1.0,0.862,0.052,1.0
3,15.0,1.0,0.888,0.046,1.0


In [15]:
# Print 10 best and 10 worst examples at a chosen SNR
import random
import pandas as pd

pd.set_option('display.max_colwidth', None)

TARGET_SNR_FOR_EXAMPLES = 10.0
EXAMPLE_POOL_SIZE = 200
EXAMPLE_MAX_LEN = 72
EXAMPLE_SEED = 2026

rng_examples = random.Random(EXAMPLE_SEED)
pool = val_texts if len(val_texts) > 0 else sentences
sample_n = min(EXAMPLE_POOL_SIZE, len(pool))
sample_texts = rng_examples.sample(pool, sample_n)

smoothie = SmoothingFunction().method1
rows_examples = []

for ref_text in sample_texts:
    pred_text = llm_semcomm.generate_text(ref_text, snr_db=TARGET_SNR_FOR_EXAMPLES, max_len=EXAMPLE_MAX_LEN)

    ref_tokens = ref_text.split()
    hyp_tokens = pred_text.split()
    bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothie) if hyp_tokens else 0.0
    err = wer([ref_text], [pred_text])

    rows_examples.append({
        'reference_text': ref_text,
        'predicted_text': pred_text,
        'bleu': float(bleu),
        'wer': float(err),
        'exact_match': ref_text.strip() == pred_text.strip(),
        'ref_len': len(ref_tokens),
        'pred_len': len(hyp_tokens),
    })

df_examples = pd.DataFrame(rows_examples)

best_5 = (
    df_examples
    .sort_values(by=['wer', 'bleu'], ascending=[True, False])
    .head(5)
    .reset_index(drop=True)
)

worst_5 = (
    df_examples
    .sort_values(by=['wer', 'bleu'], ascending=[False, True])
    .head(5)
    .reset_index(drop=True)
)

print(f'Best 5 examples @ SNR={TARGET_SNR_FOR_EXAMPLES} dB')
display(best_5)
print(f'Worst 5 examples @ SNR={TARGET_SNR_FOR_EXAMPLES} dB')
display(worst_5)

Best 5 examples @ SNR=10.0 dB


,reference_text,predicted_text,bleu,wer,exact_match,ref_len,pred_len
0,The debate is closed.,The debate is closed.,1.0,0.0,True,4,4
1,The debate is closed.,The debate is closed.,1.0,0.0,True,4,4
2,The debate is closed.,The debate is closed.,1.0,0.0,True,4,4
3,Waste management is a very complex issue subject to extensive lobbying. This is why the European Union has adopted a comprehensive strategy to tackle this major environmental and health challenge.,Waste management is a very complex issue subject to extensive lobbying. This is why the European Union has adopted a comprehensive strategy to tackle this major environmental and health challenge.,1.0,0.0,True,30,30
4,"Thank you, Mrs Laguiller. I will give the letter my utmost attention.","Thank you, Mrs Laguiller. I will give the letter my utmost attention.",1.0,0.0,True,12,12


Worst 5 examples @ SNR=10.0 dB


,reference_text,predicted_text,bleu,wer,exact_match,ref_len,pred_len
0,"Consumer welfare demands that we use every means possible to prevent and vigorously combat these diseases and their vectors. The action taken in 1997 is now proving even more inadequate as scientists are today talking about a third transmission route. We must therefore provide ourselves with effective means to eradicate transmissible spongiform encephalopathies once and for all, if at all possible. This means coordinating and dealing with the sector as soon as waste is used in animal fodder, destroying at-risk materials if there is suspicion, banning the inclusion of animal protein in cattle fodder, demanding traceability by labelling the animal' s site of birth and rearing, and, at Member State level, setting up the local monitoring systems which are the only ways to guarantee that the campaign will be effective and that monitoring will be transparent, setting in place a fast, systematic screening procedure and also in-depth checks, if necessary. Finally, the whole herd must be slaughtered if one beast is found to be infected. This is the price of food safety, but it must not be used as an excuse to cast doubt on the transportation of live animals.",Consumer welfare demands that we use every means possible to prevent and vigorously combat these diseases and their vectors once and,0.000282,0.894737,False,190,21
1,"The European Commission has made this proposal supposedly in a spirit of consensus but, frankly, having analysed it at length with the best will in the world, I cannot find a single reason to accept it. I take comfort in the fact that I am not the only one. The Commission' s proposal has not been well received by anyone, neither by the Community producers, nor by the ACP countries. It seems that the Council cannot easily accept it. Parliament does not like it either. What is even more serious however is that not even the United States like it.","I take comfort in the fact that I am not the only one who is not well received by anyone in any way of consensus. I have supposedly made this proposal in a spirit of consensus but, frankly, have analysed it at length with the best will in the world, I cannot find a single reason to accept it. The Commission'",0.339728,0.850000,False,100,61
2,"Mr President, the report is concerned with supplementing a list of permitted additives with additional foodstuff additives, other than colouring agents and flavourings. This means that the Commission suggests supplementing the list with additives which have not yet been used. It does not, however, mean deleting any additives from the list. In terms of food and food safety, the needs and wishes of the consumer need to be given more consideration than has been the case in the past. That does not mean, of course, that we can ignore the interests of the foodstuff manufacturers or that we should turn our back on modern food production methods, as Mr Pojarno mentioned a moment ago. It does mean, however, that food safety must be at the top of the agenda. Whilst many additives are harmless, this does not mean that they all are. And if harmlessness has not been established 100%, there is no doubt that the only correct way forward is to ban the use of the product. In addition, it is useful to get manufacturers to spell out the benefit of adding substances for the consumer. The example of E401 or sodium alginate - a word which I can pronounce - has been mentioned. This makes stale carrots look fresh. What consumer interest is served by doing this? What is in the interests of the consumer differs from one consumer to another and from one consumer group to the next. But what is clear is that all consumers benefit from sound consumer information so that they are not misled. This report literally takes a refreshing look at the problem, and not before time. The rapporteur is to be congratulated on this and the report can count on the support of the liberal group.","Mr President, th

In [ ]:
# Aggregate benchmark with automatic 2-sentence precheck (time-saving + safer runs)
import numpy as np
import pandas as pd
import time

pd.set_option('display.max_colwidth', None)

# -------------------------
# Run mode presets
# -------------------------
# quick: very fast smoke benchmark
# mentor_full_200: your mentor-facing full test
RUN_MODE = 'mentor_full_200'  # change to 'quick' if needed

if RUN_MODE == 'quick':
    EVAL_SENT_COUNT = 60
    GEN_MAX_LEN = 64
else:
    EVAL_SENT_COUNT = 200
    GEN_MAX_LEN = 72

SNR_EVAL_LIST = [0.0, 5.0, 10.0, 15.0]
SEED_EVAL = 123
PRECHECK_COUNT = 2
rng = random.Random(SEED_EVAL)

eval_pool = val_texts if len(val_texts) > 0 else sentences
full_count = min(EVAL_SENT_COUNT, len(eval_pool))
if full_count == 0:
    raise ValueError('No evaluation sentences available.')

eval_samples = rng.sample(eval_pool, full_count)
precheck_samples = eval_samples[:min(PRECHECK_COUNT, full_count)]

smoothie = SmoothingFunction().method1
start_all = time.time()

# -------------------------
# Step 1: tiny precheck
# -------------------------
print(f'Precheck: running {len(precheck_samples)} sentences at 10 dB before full benchmark...')
precheck_ok = True

for idx, ref_text in enumerate(precheck_samples, start=1):
    try:
        pred_text = llm_semcomm.generate_text(ref_text, snr_db=10.0, max_len=GEN_MAX_LEN)
        print(f'Precheck {idx}/{len(precheck_samples)} OK | ref_len={len(ref_text.split())} | pred_len={len(pred_text.split())}')
    except Exception as ex:
        precheck_ok = False
        print(f'Precheck failed at sample {idx}: {type(ex).__name__} - {ex}')
        break

if not precheck_ok:
    raise RuntimeError('Stopped before full benchmark because precheck failed.')

print('Precheck passed. Starting full benchmark...')

# -------------------------
# Step 2: full benchmark
# -------------------------
results_many = []
try:
    for snr_db in SNR_EVAL_LIST:
        bleu_list = []
        wer_list = []
        exact_matches = 0
        snr_start = time.time()

        for idx, ref_text in enumerate(eval_samples, start=1):
            pred_text = llm_semcomm.generate_text(ref_text, snr_db=snr_db, max_len=GEN_MAX_LEN)

            ref_tokens = ref_text.split()
            hyp_tokens = pred_text.split()
            bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothie) if hyp_tokens else 0.0
            err = wer([ref_text], [pred_text])

            bleu_list.append(float(bleu))
            wer_list.append(float(err))
            if ref_text.strip() == pred_text.strip():
                exact_matches += 1

            if idx % 25 == 0 or idx == full_count:
                print(f'SNR {snr_db:>4} dB | processed {idx}/{full_count}')

        results_many.append({
            'run_mode': RUN_MODE,
            'snr_db': snr_db,
            'n_sent': full_count,
            'gen_max_len': GEN_MAX_LEN,
            'bleu_mean': round(float(np.mean(bleu_list)), 4),
            'bleu_std': round(float(np.std(bleu_list)), 4),
            'wer_mean': round(float(np.mean(wer_list)), 4),
            'wer_std': round(float(np.std(wer_list)), 4),
            'exact_match_rate': round(exact_matches / max(full_count, 1), 4),
            'snr_runtime_sec': round(time.time() - snr_start, 1),
        })
except KeyboardInterrupt:
    print('Evaluation interrupted by user. Showing partial results...')

df_results = pd.DataFrame(results_many)
display(df_results)
print(f'Total runtime (sec): {round(time.time() - start_all, 1)}')

Precheck: running 2 sentences at 10 dB before full benchmark...
Precheck 1/2 OK | ref_len=18 | pred_len=17
Precheck 2/2 OK | ref_len=50 | pred_len=50
Precheck passed. Starting full benchmark...
SNR  0.0 dB | processed 25/200
SNR  0.0 dB | processed 50/200
SNR  0.0 dB | processed 75/200
SNR  0.0 dB | processed 100/200
SNR  0.0 dB | processed 125/200
SNR  0.0 dB | processed 150/200
SNR  0.0 dB | processed 175/200
SNR  0.0 dB | processed 200/200
